Configuration

In [1]:
import numpy as np
import pandas as pd

RANDOM_SEED = 42

COUNTIES = [
    "Kisumu", "Siaya", "Homa Bay", "Migori",
    "Kakamega", "Bungoma", "Busia", "Vihiga",
]
N_COUNTIES = len(COUNTIES)

WAVES = [2015, 2020]
REAL_OBSERVED_CASES = {
    #              2015  2020
    "Bungoma":    (25,   53),
    "Kakamega":   (27,   29),
    "Busia":      (39,   40),
    "Siaya":      (40,   44),
    "Vihiga":     (14,   12),
    "Kisumu":     (22,   23),
    "Homa Bay":   (27,   10),
    "Migori":     (19,   23),
}

COUNTY_PROFILES = {
    "Kisumu":   dict(p_case=0.38, p_net=0.55, p_spray=0.22, p_pf=0.30, p_res=0.95, Age=(5,110),temp=(22, 31), rain=(1, 9)),
    "Siaya":    dict(p_case=0.42, p_net=0.50, p_spray=0.18, p_pf=0.33, p_res=0.96, Age=(5,110), temp=(21, 30), rain=(1, 8)),
    "Homa Bay": dict(p_case=0.40, p_net=0.48, p_spray=0.20, p_pf=0.31, p_res=0.95,Age=(5,110), temp=(21, 31), rain=(1, 9)),
    "Migori":   dict(p_case=0.35, p_net=0.52, p_spray=0.19, p_pf=0.28, p_res=0.94,Age=(5,110), temp=(20, 30), rain=(2, 10)),
    "Kakamega": dict(p_case=0.28, p_net=0.60, p_spray=0.25, p_pf=0.22, p_res=0.93,Age=(5,110), temp=(19, 28), rain=(2, 10)),
    "Bungoma":  dict(p_case=0.25, p_net=0.58, p_spray=0.24, p_pf=0.20, p_res=0.92,Age=(5,110), temp=(18, 28), rain=(2, 9)),
    "Busia":    dict(p_case=0.33, p_net=0.53, p_spray=0.21, p_pf=0.27, p_res=0.94,Age=(5,110),temp=(20, 29), rain=(1, 9)),
    "Vihiga":   dict(p_case=0.22, p_net=0.62, p_spray=0.26, p_pf=0.18, p_res=0.91,Age=(5,110),temp=(18, 27), rain=(2, 9)),
}


N_PER_COUNTY_WAVE = 1500 
def _calibrate_spatiotemporal_effects():
    counties = ["Kisumu", "Siaya", "Homa Bay", "Migori",
                "Kakamega", "Bungoma", "Busia", "Vihiga"]
    target_logit = np.zeros((len(counties), 2))
    for i, c in enumerate(counties):
        for w, cnt in enumerate(REAL_OBSERVED_CASES[c]):
            p = cnt / N_PER_COUNTY_WAVE
            target_logit[i, w] = np.log(p / (1 - p))
    intercept = float(target_logit.mean())
    u = target_logit.mean(axis=1) - intercept
    gamma = target_logit.mean(axis=0) - intercept
    return intercept, dict(zip(counties, u)), dict(zip([2015, 2020], gamma))

CALIBRATED_INTERCEPT, CALIBRATED_U, CALIBRATED_GAMMA = _calibrate_spatiotemporal_effects()

# Bayesian estimation (non-reversible MH-style sampler)
N_CHAINS = 3
N_ITER = 10000
BURN_IN = 1000
STEP_SIZE_BETA = 0.05
STEP_SIZE_SIGMA2 = 0.05
FLIP_PROB_Q = 0.15            # probability of flipping auxiliary direction phi
PRIOR_MU = 0.0
PRIOR_TAU2 = 25.0              # non-informative prior variance for beta mean
PRIOR_ALPHA = 2.0              # inverse-gamma shape for sigma2
PRIOR_LAMBDA = 1.0             # inverse-gamma scale for sigma2
SPATIAL_SIGMA2_U = 0.25        # variance of spatial random effect u_i
TEMPORAL_SIGMA2_GAMMA = 0.15   # variance of temporal effect gamma_t

COVARIATE_NAMES = [
    "net_use", "spray_use",
    "Age", "wealth_middle", "wealth_richer", "wealth_richest",   
    "educ_primary", "educ_secondary", "educ_tertiary", 
    "temperature", "rainfall",
]
N_BETA = len(COVARIATE_NAMES)
N_WEALTH_LEVELS = 4
N_EDUC_LEVELS = 4
TRUE_BETA = np.array([
    -0.9, -0.7,                # net_use, spray_use
    0.31,                      # Age
    -0.10, -0.20, -0.35,       # wealth_middle, wealth_richer, wealth_richest
    -0.12, -0.22, -0.30,       # educ_primary, educ_secondary, educ_tertiary
    0.05, 0.04,                # temperature, rainfall
])

# Reinforcement learning (5-critic min-max ensemble TD3)
N_CRITICS = 5
STATE_DIM = 1 + N_COUNTIES
ACTION_DIM = 2                 # (spraying rate, net-use rate)
ACTOR_LR = 1e-3
CRITIC_LR = 1e-3
GAMMA = 0.99
TAU = 0.005                    # soft update coefficient (delta)
POLICY_NOISE = 0.15
NOISE_CLIP = 0.3
EXPLORATION_NOISE = 0.15
POLICY_DELAY = 2
BATCH_SIZE = 64
REPLAY_CAPACITY = 50_000

GRID_MIN, GRID_MAX = 0.0, 20.0
HUMAN_SPEED = 2.0
VECTOR_SPEED = 0.8             # slower than the human agent
INTERACTION_THRESHOLD = 6.5    # normalized proximity threshold
                                
BASE_TRANSMISSION_RATE = 0.35  # rho: probability of transmission given proximity
EPISODE_MAX_STEPS = 60         # ~ "avoid interaction for 30s" horizon (2 checks/sec)

EFFECT_SPRAY = 0.42        # asymptotic max protective effect of spraying
EFFECT_NET = 0.58          # asymptotic max protective effect of net use
K_SPRAY = 2.5               # saturation rate (how fast spraying's effect saturates)
K_NET = 3.0                 # saturation rate for net use
COST_SPRAY = 0.006
COST_NET = 0.004

BOUNDARY_BARRIER = 0.06

N_TRAIN_EPISODES = 100000
EVAL_EVERY = 300
N_EVAL_EPISODES = 20

# Evaluation rollout for predicted case counts
CASE_EVAL_STEPS = 1200

Bayesian Model

In [2]:
rng_global = np.random.default_rng(RANDOM_SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))
# ----------------------------------------------------------------------
# 1. Location and wave-indexed data simulation
# ----------------------------------------------------------------------
def simulate_data(seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)

    true_u = dict(CALIBRATED_U)
    true_gamma = dict(CALIBRATED_GAMMA)
    true_intercept = CALIBRATED_INTERCEPT

    rows = []
    for county in COUNTIES:
        prof = COUNTY_PROFILES[county]
        for wave in WAVES:
            n = N_PER_COUNTY_WAVE
            net_use = rng.binomial(1, prof["p_net"], n)
            spray_use = rng.binomial(1, prof["p_spray"], n)
            Age = rng.uniform(prof["Age"][0], prof["Age"][1], n)
            wealth = rng.integers(1, N_WEALTH_LEVELS + 1, n)
            education = rng.integers(1, N_EDUC_LEVELS + 1, n)

            temperature = rng.uniform(prof["temp"][0], prof["temp"][1], n)
            rainfall = rng.uniform(prof["rain"][0], prof["rain"][1], n)

            df = pd.DataFrame({
                "net_use": net_use,
                "spray_use": spray_use,
                "Age": Age,
                # wealth (levels 2,3,4 vs. reference level 1)
                "wealth_middle": (wealth == 2).astype(float),
                "wealth_richer": (wealth == 3).astype(float),
                "wealth_richest": (wealth == 4).astype(float),
                # education (levels 2,3,4 vs. reference level 1)
                "educ_primary": (education == 2).astype(float),
                "educ_secondary": (education == 3).astype(float),
                "educ_tertiary": (education == 4).astype(float),
                "temperature": temperature,
                "rainfall": rainfall,
            })

            # standardize continuous covariates
            for col in ("Age", "temperature", "rainfall"):
                df[col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)

            S_std = df[COVARIATE_NAMES].values
            S_centered = S_std.copy()
            for j, name in enumerate(COVARIATE_NAMES):
                S_centered[:, j] = S_centered[:, j] - S_centered[:, j].mean()

            eps = rng.normal(0, 0.35, n)
            covariate_beta = TRUE_BETA.copy()
            eta = (
                S_centered @ covariate_beta
                + true_intercept + true_u[county] + true_gamma[wave]
                + eps
            )
            p = sigmoid(eta)
            y = rng.binomial(1, p)

            out = pd.DataFrame(S_std, columns=COVARIATE_NAMES)
            out["y"] = y
            out["county"] = county
            out["wave"] = wave
            out["county_idx"] = COUNTIES.index(county)
            out["wave_idx"] = WAVES.index(wave)
            rows.append(out)

    data = pd.concat(rows, ignore_index=True)

    observed_cases = data.groupby("county")["y"].sum().reindex(COUNTIES)
    return data, true_u, true_gamma, observed_cases


def real_observed_cases_by_wave():
    rows = []
    for county in COUNTIES:
        c2015, c2020 = REAL_OBSERVED_CASES[county]
        rows.append(dict(county=county, cases_2015=c2015, cases_2020=c2020,
                          cases_total=c2015 + c2020))
    df = pd.DataFrame(rows).set_index("county").reindex(COUNTIES)
    return df


# ----------------------------------------------------------------------
# 2. Non-reversible-style MH sampler for beta_j, Gibbs for sigma2_j,
#    MH for u_i and gamma_t
# ----------------------------------------------------------------------
def log_likelihood(y, S, beta, u_vec, gamma_vec, county_idx, wave_idx):
    eta = S @ beta + u_vec[county_idx] + gamma_vec[wave_idx]
    p = sigmoid(eta)
    p = np.clip(p, 1e-8, 1 - 1e-8)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def log_prior_beta(beta, mu, tau2):
    return -0.5 * np.sum((beta - mu) ** 2) / tau2

def run_chain(data, chain_id, seed):
    rng = np.random.default_rng(seed + 1000 * chain_id)
    y = data["y"].values
    S = data[COVARIATE_NAMES].values
    county_idx = data["county_idx"].values
    wave_idx = data["wave_idx"].values

    beta = rng.normal(0, 0.5, N_BETA)
    phi = rng.choice([-1.0, 1.0], size=N_BETA)
    sigma2_j = np.full(N_BETA, 1.0)
    u_vec = rng.normal(0, 0.3, N_COUNTIES)
    gamma_vec = rng.normal(0, 0.3, len(WAVES))

    beta_trace = np.zeros((N_ITER, N_BETA))
    sigma2_trace = np.zeros((N_ITER, N_BETA))
    u_trace = np.zeros((N_ITER, N_COUNTIES))
    gamma_trace = np.zeros((N_ITER, len(WAVES)))
    accept_beta = np.zeros(N_BETA)
    accept_u = 0
    accept_gamma = 0

    for it in range(N_ITER):
        # beta_j: non-reversible-style MH, one coordinate at a time
        for j in range(N_BETA):
            beta_prop = beta.copy()
            beta_prop[j] = beta[j] + phi[j] * STEP_SIZE_BETA

            ll_curr = log_likelihood(y, S, beta, u_vec, gamma_vec, county_idx, wave_idx)
            ll_prop = log_likelihood(y, S, beta_prop, u_vec, gamma_vec, county_idx, wave_idx)
            lp_curr = -0.5 * (beta[j] - PRIOR_MU) ** 2 / sigma2_j[j]
            lp_prop = -0.5 * (beta_prop[j] - PRIOR_MU) ** 2 / sigma2_j[j]

            log_alpha = (ll_prop + lp_prop) - (ll_curr + lp_curr)
            if np.log(rng.uniform()) < log_alpha:
                beta[j] = beta_prop[j]
                accept_beta[j] += 1
            else:
                phi[j] = -phi[j]

            # stochastic direction flip (auxiliary variable update)
            if rng.uniform() < FLIP_PROB_Q:
                phi[j] = -phi[j]

        # sigma2_j: exact Gibbs draw from Inverse-Gamma
        alpha_post = PRIOR_ALPHA + 0.5
        lambda_post = PRIOR_LAMBDA + 0.5 * (beta - PRIOR_MU) ** 2
        sigma2_j = 1.0 / rng.gamma(alpha_post, 1.0 / lambda_post)

        # u_i: random-walk MH per county against CAR-style prior
        for i in range(N_COUNTIES):
            mask = county_idx == i
            neighbor_mean = (u_vec.sum() - u_vec[i]) / (N_COUNTIES - 1)
            u_prop = u_vec.copy()
            u_prop[i] = u_vec[i] + rng.normal(0, 0.05)

            ll_curr = log_likelihood(y[mask], S[mask], beta, u_vec, gamma_vec,
                                      county_idx[mask], wave_idx[mask])
            ll_prop = log_likelihood(y[mask], S[mask], beta, u_prop, gamma_vec,
                                      county_idx[mask], wave_idx[mask])
            lp_curr = -0.5 * (u_vec[i] - neighbor_mean) ** 2 / SPATIAL_SIGMA2_U
            lp_prop = -0.5 * (u_prop[i] - neighbor_mean) ** 2 / SPATIAL_SIGMA2_U

            log_alpha = (ll_prop + lp_prop) - (ll_curr + lp_curr)
            if np.log(rng.uniform()) < log_alpha:
                u_vec[i] = u_prop[i]
                accept_u += 1
        u_vec -= u_vec.mean()

        # gamma_t: random-walk MH per wave
        for t in range(len(WAVES)):
            mask = wave_idx == t
            gamma_prop = gamma_vec.copy()
            gamma_prop[t] = gamma_vec[t] + rng.normal(0, 0.05)

            ll_curr = log_likelihood(y[mask], S[mask], beta, u_vec, gamma_vec,
                                      county_idx[mask], wave_idx[mask])
            ll_prop = log_likelihood(y[mask], S[mask], beta, u_vec, gamma_prop,
                                      county_idx[mask], wave_idx[mask])
            lp_curr = -0.5 * gamma_vec[t] ** 2 / TEMPORAL_SIGMA2_GAMMA
            lp_prop = -0.5 * gamma_prop[t] ** 2 / TEMPORAL_SIGMA2_GAMMA

            log_alpha = (ll_prop + lp_prop) - (ll_curr + lp_curr)
            if np.log(rng.uniform()) < log_alpha:
                gamma_vec[t] = gamma_prop[t]
                accept_gamma += 1

        gamma_vec -= gamma_vec.mean()

        beta_trace[it] = beta
        sigma2_trace[it] = sigma2_j
        u_trace[it] = u_vec
        gamma_trace[it] = gamma_vec

    return dict(
        beta=beta_trace, sigma2=sigma2_trace, u=u_trace, gamma=gamma_trace,
        accept_rate_beta=accept_beta / N_ITER,
        accept_rate_u=accept_u / (N_ITER * N_COUNTIES),
        accept_rate_gamma=accept_gamma / (N_ITER * len(WAVES)),
    )

def run_all_chains(data):
    chains = [run_chain(data, c, RANDOM_SEED) for c in range(N_CHAINS)]
    return chains

# ----------------------------------------------------------------------
# 3. Convergence diagnostics
# ----------------------------------------------------------------------
def gelman_rubin(chains_param):
    M, N = chains_param.shape
    chain_means = chains_param.mean(axis=1)
    grand_mean = chain_means.mean()
    B = N / (M - 1) * np.sum((chain_means - grand_mean) ** 2)
    W = np.mean(np.var(chains_param, axis=1, ddof=1))
    var_hat = (1 - 1 / N) * W + B / N
    if W <= 0:
        return 1.0
    return float(np.sqrt(var_hat / W))


def autocorrelation(x, lag):
    x = x - x.mean()
    if lag == 0:
        return 1.0
    num = np.sum(x[:-lag] * x[lag:])
    den = np.sum(x ** 2)
    return num / den if den > 0 else 0.0


def effective_sample_size(x, max_lag=100):
    n = len(x)
    rho_sum = 0.0
    for k in range(1, min(max_lag, n - 1)):
        rho_k = autocorrelation(x, k)
        if rho_k < 0.05:
            break
        rho_sum += rho_k
    ess = n / (1 + 2 * rho_sum)
    return max(ess, 1.0)


def monte_carlo_se(x):
    ess = effective_sample_size(x)
    return float(np.std(x, ddof=1) / np.sqrt(ess))


def compute_diagnostics(chains, burn_in=BURN_IN):
    beta_post = np.stack([c["beta"][burn_in:] for c in chains])
    rows = []
    for j, name in enumerate(COVARIATE_NAMES):
        param_chains = beta_post[:, :, j]
        rhat = gelman_rubin(param_chains)
        pooled = param_chains.reshape(-1)
        ess = effective_sample_size(pooled)
        mcse = monte_carlo_se(pooled)
        ac1 = autocorrelation(pooled, 1)
        rows.append(dict(
            parameter=name, mean=pooled.mean(), sd=pooled.std(ddof=1),
            gelman_rubin_Rhat=rhat, ess=ess, mcse=mcse, autocorr_lag1=ac1,
        ))
    return pd.DataFrame(rows)

def posterior_predictive_check(data, beta_hat, u_hat, gamma_hat, n_rep=200, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    S = data[COVARIATE_NAMES].values
    county_idx = data["county_idx"].values
    wave_idx = data["wave_idx"].values
    eta = S @ beta_hat + u_hat[county_idx] + gamma_hat[wave_idx]
    p = sigmoid(eta)

    observed_T = data["y"].mean()
    rep_T = np.zeros(n_rep)
    for r in range(n_rep):
        y_rep = rng.binomial(1, p)
        rep_T[r] = y_rep.mean()

    p_bayes = float(np.mean(rep_T >= observed_T))
    return observed_T, rep_T, p_bayes


def fit_pi_it(data, beta_hat, u_hat, gamma_hat):
    pi_lookup = {}
    for i, county in enumerate(COUNTIES):
        for t, wave in enumerate(WAVES):
            subset = data[(data["county"] == county) & (data["wave"] == wave)]
            s_bar = subset[COVARIATE_NAMES].values.mean(axis=0)
            eta_bar = s_bar @ beta_hat + u_hat[i] + gamma_hat[t]
            pi_lookup[(county, wave)] = float(sigmoid(eta_bar))
    return pi_lookup


rl_env

In [3]:
_COUNTY_EFFECTIVE_COST = {
    county: (
        COST_SPRAY / COUNTY_PROFILES[county]["p_spray"],
        COST_NET / COUNTY_PROFILES[county]["p_net"],
    )
    for county in COUNTIES
}


class DiseaseSpreadEnv:
    def __init__(self, pi_lookup, wave=2020, intervention_cost=0.02, seed=None):
        self.pi_lookup = pi_lookup
        self.wave = wave
        self.intervention_cost = intervention_cost
        self.rng = np.random.default_rng(seed)
        self.county_idx = None
        self.human_pos = None
        self.vector_pos = None
        self.t = 0
        self.episode_cases = 0

    @property
    def state_dim(self):
        return 1 + N_COUNTIES

    @property
    def action_dim(self):
        return 2

    def _one_hot(self, idx):
        v = np.zeros(N_COUNTIES)
        v[idx] = 1.0
        return v

    def reset(self, county_idx=None):
        if county_idx is None:
            county_idx = self.rng.integers(0, N_COUNTIES)
        self.county_idx = county_idx
        self.human_pos = self.rng.uniform(GRID_MIN, GRID_MAX, 2)
        self.vector_pos = self.rng.uniform(GRID_MIN, GRID_MAX, 2)
        self.t = 0
        self.episode_cases = 0
        interaction_flag = 0.0
        state = np.concatenate([[interaction_flag], self._one_hot(county_idx)])
        return state.astype(np.float32)

    def _pi_it(self):
        county = COUNTIES[self.county_idx]
        return self.pi_lookup[(county, self.wave)]

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)
        a_spray, a_net = action

        # random movement
        self.human_pos = np.clip(
            self.human_pos + self.rng.uniform(-1, 1, 2) * HUMAN_SPEED,
            GRID_MIN, GRID_MAX,
        )
        self.vector_pos = np.clip(
            self.vector_pos + self.rng.uniform(-1, 1, 2) * VECTOR_SPEED,
            GRID_MIN, GRID_MAX,
        )

        d_t = np.linalg.norm(self.human_pos - self.vector_pos)
        proximity = d_t < INTERACTION_THRESHOLD

        case = False
        if proximity:
            protective = (
                EFFECT_SPRAY * (1.0 - np.exp(-K_SPRAY * a_spray))
                + EFFECT_NET * (1.0 - np.exp(-K_NET * a_net))
            )
            p_eff = BASE_TRANSMISSION_RATE * (1.0 - protective)
            p_eff = float(np.clip(p_eff, 0.0, 1.0))
            case = bool(self.rng.uniform() < p_eff)

        pi_it = self._pi_it()
        r_t = -1.0 if case else 1.0
        cost_spray_eff, cost_net_eff = _COUNTY_EFFECTIVE_COST[COUNTIES[self.county_idx]]
        reward = (
            r_t * pi_it
            - cost_spray_eff * a_spray ** 2 - cost_net_eff * a_net ** 2
            - BOUNDARY_BARRIER * (a_spray ** 8 + a_net ** 8)
        )

        if case:
            self.episode_cases += 1

        self.t += 1
        done = self.t >= EPISODE_MAX_STEPS

        interaction_flag = 1.0 if case else 0.0
        next_state = np.concatenate(
            [[interaction_flag], self._one_hot(self.county_idx)]
        ).astype(np.float32)

        info = dict(case=case, distance=d_t, county=COUNTIES[self.county_idx], pi_it=pi_it)
        return next_state, float(reward), done, info


td3_agent

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random

torch.manual_seed(RANDOM_SEED)
device = torch.device("cpu")


class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
        nn.init.uniform_(self.net[-1].weight, -3e-3, 3e-3)
        nn.init.constant_(self.net[-1].bias, -2.5)

    def forward(self, state):
        # sigmoid bounds actions to (0,1), matching a_spraying, a_net in (0,1)
        return torch.sigmoid(self.net(state))


class Critic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1))


class ReplayBuffer:
    def __init__(self, capacity=REPLAY_CAPACITY):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buffer.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return (
            torch.tensor(s, dtype=torch.float32),
            torch.tensor(a, dtype=torch.float32),
            torch.tensor(r, dtype=torch.float32).unsqueeze(-1),
            torch.tensor(s2, dtype=torch.float32),
            torch.tensor(d, dtype=torch.float32).unsqueeze(-1),
        )

    def __len__(self):
        return len(self.buffer)


def min_max_ensemble(q_list):
    q1, q2, q3, q4, q5 = q_list
    inner_min = torch.min(torch.min(q2, q3), q4)
    q_hat = torch.max(torch.max(inner_min, q5), q1)
    return q_hat


class TD3EnsembleAgent:
    def __init__(self, state_dim, action_dim):
        self.actor = Actor(state_dim, action_dim).to(device)
        self.actor_target = Actor(state_dim, action_dim).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=ACTOR_LR)

        self.critics = [Critic(state_dim, action_dim).to(device) for _ in range(N_CRITICS)]
        self.critic_targets = [Critic(state_dim, action_dim).to(device) for _ in range(N_CRITICS)]
        for c, ct in zip(self.critics, self.critic_targets):
            ct.load_state_dict(c.state_dict())
        self.critic_opts = [torch.optim.Adam(c.parameters(), lr=CRITIC_LR) for c in self.critics]

        self.replay = ReplayBuffer()
        self.total_it = 0
        self.action_dim = action_dim

    def select_action(self, state, noise_scale=0.0):
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            action = self.actor(state_t).cpu().numpy()[0]
        if noise_scale > 0:
            action = action + np.random.normal(0, noise_scale, size=self.action_dim)
        return np.clip(action, 0.0, 1.0)

    def train_step(self):
        if len(self.replay) < BATCH_SIZE:
            return None
        self.total_it += 1
        s, a, r, s2, d = self.replay.sample(BATCH_SIZE)

        with torch.no_grad():
            noise = (torch.randn_like(a) * POLICY_NOISE).clamp(-NOISE_CLIP, NOISE_CLIP)
            a2 = (self.actor_target(s2) + noise).clamp(0.0, 1.0)
            q_targets = [ct(s2, a2) for ct in self.critic_targets]
            q_hat = min_max_ensemble(q_targets)
            y = r + GAMMA * (1 - d) * q_hat

        critic_loss_val = 0.0
        for critic, opt in zip(self.critics, self.critic_opts):
            q = critic(s, a)
            loss = F.mse_loss(q, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            critic_loss_val += loss.item()

        actor_loss_val = None
        if self.total_it % POLICY_DELAY == 0:
            actor_loss = -self.critics[0](s, self.actor(s)).mean()
            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()
            actor_loss_val = actor_loss.item()

            # soft (Polyak) update of target networks
            for critic, ct in zip(self.critics, self.critic_targets):
                for p, tp in zip(critic.parameters(), ct.parameters()):
                    tp.data.copy_(TAU * p.data + (1 - TAU) * tp.data)
            for p, tp in zip(self.actor.parameters(), self.actor_target.parameters()):
                tp.data.copy_(TAU * p.data + (1 - TAU) * tp.data)

        return dict(critic_loss=critic_loss_val / N_CRITICS, actor_loss=actor_loss_val)


Training

In [5]:
def run_episode(env, agent, county_idx=None, noise_scale=0.0, train=True):
    state = env.reset(county_idx=county_idx)
    ep_reward = 0.0
    ep_cases = 0
    ep_steps = 0
    for _ in range(EPISODE_MAX_STEPS):
        action = agent.select_action(state, noise_scale=noise_scale)
        next_state, reward, done, info = env.step(action)
        if train:
            agent.replay.push(state, action, reward, next_state, float(done))
            agent.train_step()
        ep_reward += reward
        ep_cases += int(info["case"])
        ep_steps += 1
        state = next_state
        if done:
            break
    return ep_reward, ep_cases, ep_steps


def evaluate(env, agent, n_episodes=N_EVAL_EPISODES):
    successes = 0
    total_cases = 0
    total_episodes = 0
    total_steps = 0
    for county_idx in range(N_COUNTIES):
        for _ in range(n_episodes):
            ep_reward, ep_cases, ep_steps = run_episode(
                env, agent, county_idx=county_idx, noise_scale=0.0, train=False,
            )
            total_episodes += 1
            total_steps += ep_steps
            total_cases += ep_cases
            if ep_cases == 0:
                successes += 1
    success_rate = successes / total_episodes * 100
    infection_rate = total_cases / total_steps * 100
    return success_rate, infection_rate


def total_predicted_cases_snapshot(env, agent, n_steps=150, seed=0):
    rng = np.random.default_rng(seed)
    total_cases = 0
    for i in range(N_COUNTIES):
        env.rng = rng
        state = env.reset(county_idx=i)
        for _ in range(n_steps):
            action = agent.select_action(state, noise_scale=0.0)
            state, reward, done, info = env.step(action)
            total_cases += int(info["case"])
            if done:
                state = env.reset(county_idx=i)
    return total_cases


def train_agent(env, agent, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    history = dict(episode=[], reward=[], success_rate=[], infection_rate=[],
                    total_predicted_cases=[])
    reward_trace = []
    noise_floor = 0.02
    sr0, ir0 = evaluate(env, agent, n_episodes=5)
    cases0 = total_predicted_cases_snapshot(env, agent)
    history["episode"].append(0)
    history["reward"].append(np.nan)
    history["success_rate"].append(sr0)
    history["infection_rate"].append(ir0)
    history["total_predicted_cases"].append(cases0)
    print(f"[ep     0] (untrained baseline) success_rate={sr0:.1f}% "
          f"infection_rate={ir0:.2f}% total_predicted_cases={cases0}")

    for ep in range(1, N_TRAIN_EPISODES + 1):
        frac = ep / N_TRAIN_EPISODES
        noise_scale = EXPLORATION_NOISE + (noise_floor - EXPLORATION_NOISE) * frac

        county_idx = int(rng.integers(0, N_COUNTIES))
        ep_reward, ep_cases, _ = run_episode(
            env, agent, county_idx=county_idx,
            noise_scale=noise_scale, train=True,
        )
        reward_trace.append(ep_reward)

        if ep % EVAL_EVERY == 0:
            sr, ir = evaluate(env, agent, n_episodes=5)
            total_cases = total_predicted_cases_snapshot(env, agent)
            history["episode"].append(ep)
            history["reward"].append(np.mean(reward_trace[-EVAL_EVERY:]))
            history["success_rate"].append(sr)
            history["infection_rate"].append(ir)
            history["total_predicted_cases"].append(total_cases)
            print(f"[ep {ep:5d}] mean_reward={history['reward'][-1]:.2f} "
                  f"success_rate={sr:.1f}% infection_rate={ir:.2f}% "
                  f"total_predicted_cases={total_cases}")

    return pd.DataFrame(history)

def learned_rates_per_county(env, agent):
    rows = []
    for i, county in enumerate(COUNTIES):
        state = np.zeros(env.state_dim, dtype=np.float32)
        state[1 + i] = 1.0
        action = agent.select_action(state, noise_scale=0.0)
        rows.append(dict(county=county, spray_rate=action[0], net_rate=action[1]))
    return pd.DataFrame(rows)

def predicted_case_counts(env, agent, real_cases_df, n_steps=CASE_EVAL_STEPS,
                           seed=RANDOM_SEED):
    rng = np.random.default_rng(seed + 7)
    rows = []
    for i, county in enumerate(COUNTIES):
        env.rng = rng
        state = env.reset(county_idx=i)
        cases = 0
        for _ in range(n_steps):
            action = agent.select_action(state, noise_scale=0.0)
            state, reward, done, info = env.step(action)
            cases += int(info["case"])
            if done:
                state = env.reset(county_idx=i)

        predicted_rate = cases / n_steps
        n_obs = N_PER_COUNTY_WAVE * len(WAVES)
        observed_total = int(real_cases_df.loc[county, "cases_total"])
        observed_rate = observed_total / n_obs
        abs_error = abs(predicted_rate - observed_rate)
        pct_error = abs_error / observed_rate * 100 if observed_rate > 0 else np.nan

        rows.append(dict(
            county=county, predicted_cases=cases, predicted_rate=predicted_rate,
            observed_cases_2015=int(real_cases_df.loc[county, "cases_2015"]),
            observed_cases_2020=int(real_cases_df.loc[county, "cases_2020"]),
            observed_cases_total=observed_total, observed_rate=observed_rate,
            case_prediction_error=abs_error, case_prediction_error_pct=pct_error,
        ))
    return pd.DataFrame(rows)

def cross_location_variance(rates_df):
    return dict(
        var_spray=float(rates_df["spray_rate"].var(ddof=0)),
        var_net=float(rates_df["net_rate"].var(ddof=0)),
        mean_spray=float(rates_df["spray_rate"].mean()),
        mean_net=float(rates_df["net_rate"].mean()),
    )


Plots

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = "outputs"

def plot_trace_beta(chains, burn_in=BURN_IN, fname=f"{OUT}/Trace_plots_beta.png"):
    n_chains = len(chains)
    ncols = 2
    nrows = int(np.ceil(N_BETA / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(11, 2.1 * nrows), sharex=True)
    axes = np.atleast_1d(axes).flatten()
    colors = plt.cm.tab10(np.linspace(0, 1, n_chains))
    for j in range(N_BETA):
        ax = axes[j]
        for c_idx, chain in enumerate(chains):
            ax.plot(chain["beta"][:, j], color=colors[c_idx], lw=0.6,
                     label=f"chain {c_idx+1}" if j == 0 else None)
        ax.axvspan(0, burn_in, color="grey", alpha=0.15)
        ax.axhline(TRUE_BETA[j], color="black", ls="--", lw=0.8)
        ax.set_ylabel(COVARIATE_NAMES[j], fontsize=8)
    for k in range(N_BETA, len(axes)):
        axes[k].axis("off")
    axes[0].legend(loc="upper right", fontsize=7, ncol=n_chains)
    for j in range(max(0, N_BETA - ncols), N_BETA):
        axes[j].set_xlabel("MCMC iteration")
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)


def plot_beta_posterior_vs_true(diagnostics_df, fname=f"{OUT}/Beta_estimation_accuracy.png"):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(N_BETA)
    ax.bar(x - 0.18, TRUE_BETA, width=0.35, label="True beta")
    ax.bar(x + 0.18, diagnostics_df["mean"].values, width=0.35,
           yerr=diagnostics_df["sd"].values, capsize=3, label="Posterior mean (+-1 SD)")
    ax.set_xticks(x)
    ax.set_xticklabels(COVARIATE_NAMES, rotation=45, ha="right")
    ax.set_ylabel("Coefficient value")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)


def plot_rmse(diagnostics_df, fname=f"{OUT}/Beta_rmse.png"):
    rmse_overall = np.sqrt(np.mean((diagnostics_df["mean"].values - TRUE_BETA) ** 2))
    per_param_err = np.abs(diagnostics_df["mean"].values - TRUE_BETA)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(COVARIATE_NAMES, per_param_err, color="firebrick")
    ax.set_xticks(range(N_BETA))
    ax.set_xticklabels(COVARIATE_NAMES, rotation=45, ha="right")
    ax.set_ylabel("|posterior mean - true beta|")
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    return rmse_overall

def plot_posterior_predictive(observed_T, rep_T, p_bayes, fname=f"{OUT}/Posterior_predictive_check.png"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(rep_T, bins=30, color="steelblue", alpha=0.8, label="Replicated T(y_rep)")
    ax.axvline(observed_T, color="black", ls="--", lw=2, label="Observed T(y)")
    ax.set_xlabel("Mean case proportion")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)

def plot_training_curves(history_df, fname=f"{OUT}/Training_curves.png"):
    fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
    axes[0].plot(history_df["episode"], history_df["reward"], color="teal")
    axes[0].set_ylabel("Mean episode reward")
    axes[1].plot(history_df["episode"], history_df["success_rate"], color="green")
    axes[1].set_ylabel("Success rate (%)")
    axes[2].plot(history_df["episode"], history_df["infection_rate"], color="crimson")
    axes[2].set_ylabel("Infection rate (%)")
    axes[2].set_xlabel("Training episode")
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)

def plot_learned_rates(rates_df, fname=f"{OUT}/Learned_intervention_rates.png"):
    x = np.arange(len(rates_df))
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(x - 0.18, rates_df["spray_rate"], width=0.35, label="Spraying rate")
    ax.bar(x + 0.18, rates_df["net_rate"], width=0.35, label="Net-use rate")
    ax.set_xticks(x)
    ax.set_xticklabels(rates_df["county"], rotation=30, ha="right")
    ax.set_ylabel("Learned prescriptive rate")
    ax.set_ylim(0, 1)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)

def plot_case_prediction(pred_df, fname=f"{OUT}/Case_prediction_error.png"):
    x = np.arange(len(pred_df))
    fig, axes = plt.subplots(2, 1, figsize=(9, 7))
    axes[0].bar(x - 0.18, pred_df["observed_rate"], width=0.35, label="Observed case rate")
    axes[0].bar(x + 0.18, pred_df["predicted_rate"], width=0.35, label="Predicted case rate")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(pred_df["county"], rotation=30, ha="right")
    axes[0].set_ylabel("Case rate")
    axes[0].legend()

    axes[1].bar(x, pred_df["case_prediction_error_pct"], color="darkorange")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(pred_df["county"], rotation=30, ha="right")
    axes[1].set_ylabel("Case Prediction Error (%)")
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)

def plot_case_reduction_over_training(history_df, fname=f"{OUT}/Case_reduction_over_training.png"):
    cases = history_df["total_predicted_cases"].values
    baseline = cases[0]
    pct_change = (cases - baseline) / baseline * 100

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(history_df["episode"], cases, color="darkred", marker="o",
            markersize=4, linewidth=2)
    ax.axhline(baseline, color="gray", ls=":", alpha=0.6,
               label=f"Untrained baseline ({baseline} cases)")
    final = cases[-1]
    final_pct = pct_change[-1]
    ax.annotate(
        f"{final_pct:+.0f}% vs. untrained baseline",
        xy=(history_df["episode"].values[-1], final),
        xytext=(0.6, 0.75), textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->", color="black"),
        fontsize=10,
    )
    ax.set_xlabel("Training episode")
    ax.set_ylabel("Total predicted cases (all 8 counties, training-time snapshot)")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)

def plot_gelman_rubin(diagnostics_df, fname=f"{OUT}/Gelman_rubin.png"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(diagnostics_df["parameter"], diagnostics_df["gelman_rubin_Rhat"], color="slateblue")
    ax.axhline(1.1, color="red", ls="--", label="R-hat = 1.1 threshold")
    ax.set_xticks(range(len(diagnostics_df)))
    ax.set_xticklabels(diagnostics_df["parameter"], rotation=45, ha="right")
    ax.set_ylabel("Gelman-Rubin R-hat")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)


Main

In [ ]:
import os
import time
import torch

os.makedirs("outputs", exist_ok=True)
def main():
    t_start = time.time()
    print("=" * 70)
    print("STEP 1: Simulating location- and wave-indexed data")
    print("=" * 70)
    data, true_u, true_gamma, observed_cases = simulate_data()
    print(f"Simulated {len(data)} observations across {N_COUNTIES} counties "
          f"and {len(WAVES)} waves.")
    observed_cases.to_csv("outputs/Observed_case_counts.csv")
    real_cases_df = real_observed_cases_by_wave()
    real_cases_df.to_csv("outputs/Real_observed_cases.csv")
    print(real_cases_df.to_string())

    print("\n" + "=" * 70)
    print("STEP 2: Bayesian estimation (non-reversible MH, 3 chains)")
    print("=" * 70)
    chains = run_all_chains(data)
    diagnostics_df = compute_diagnostics(chains)
    diagnostics_df.to_csv("outputs/Beta_convergence_diagnostics.csv", index=False)
    print(diagnostics_df.to_string(index=False))

    beta_hat = np.mean([c["beta"][BURN_IN:].mean(axis=0) for c in chains], axis=0)
    u_hat = np.mean([c["u"][BURN_IN:].mean(axis=0) for c in chains], axis=0)
    gamma_hat = np.mean([c["gamma"][BURN_IN:].mean(axis=0) for c in chains], axis=0)

    print("\nPosterior mean beta:", np.round(beta_hat, 4))
    print("True beta:          ", TRUE_BETA)

    observed_T, rep_T, p_bayes = posterior_predictive_check(data, beta_hat, u_hat, gamma_hat)
    print(f"Posterior predictive Bayesian p-value: {p_bayes:.3f}")

    print("\n Bayesian diagnostic plots")
    plot_trace_beta(chains)
    plot_beta_posterior_vs_true(diagnostics_df)
    rmse = plot_rmse(diagnostics_df)
    plot_gelman_rubin(diagnostics_df)
    plot_posterior_predictive(observed_T, rep_T, p_bayes)
    print(f"Overall beta RMSE: {rmse:.4f}")

    print("\n" + "=" * 70)
    print("STEP 3: Fitting location-and-time-varying risk surface pi_it")
    print("=" * 70)
    pi_lookup = fit_pi_it(data, beta_hat, u_hat, gamma_hat)
    for (county, wave), pi in pi_lookup.items():
        print(f"  pi[{county}, {wave}] = {pi:.4f}")
    pd.DataFrame(
        [(c, w, p) for (c, w), p in pi_lookup.items()],
        columns=["county", "wave", "pi_it"],
    ).to_csv("outputs/Fitted_pi_it.csv", index=False)

    print("\n" + "=" * 70)
    print("STEP 4: Training location-conditioned 5-critic policy")
    print("=" * 70)
    env = DiseaseSpreadEnv(pi_lookup, wave=WAVES[-1], seed=RANDOM_SEED)
    agent = TD3EnsembleAgent(env.state_dim, env.action_dim)
    history_df = train_agent(env, agent)
    history_df.to_csv("outputs/Training_history.csv", index=False)
    plot_training_curves(history_df)
    plot_case_reduction_over_training(history_df)

    print("\n" + "=" * 70)
    print("STEP 5: Evaluation")
    print("=" * 70)
    
    final_sr, final_ir = evaluate(env, agent, n_episodes=60)
    print(f"Final success rate: {final_sr:.2f}% | Final infection rate: {final_ir:.2f}%")

    rates_df = learned_rates_per_county(env, agent)
    rates_df.to_csv("outputs/Learned_intervention_rates.csv", index=False)
    print(rates_df.to_string(index=False))
    plot_learned_rates(rates_df)

    pred_df = predicted_case_counts(env, agent, real_cases_df)
    pred_df.to_csv("outputs/Case_prediction.csv", index=False)
    print(pred_df.to_string(index=False))
    plot_case_prediction(pred_df)

    adaptivity = cross_location_variance(rates_df)
    print(adaptivity)
    pd.Series(adaptivity).to_csv("outputs/Cross_location_adaptivity.csv")

    torch.save({
        "actor": agent.actor.state_dict(),
        "critics": [c.state_dict() for c in agent.critics],
    }, "outputs/td3_ensemble_checkpoint.pt")

    print(f"\nTotal runtime: {time.time() - t_start:.1f} seconds")

if __name__ == "__main__":
    main()

STEP 1: Simulating location- and wave-indexed data
Simulated 24000 observations across 8 counties and 2 waves.

Real observed case counts (calibration target):
          cases_2015  cases_2020  cases_total
county                                       
Kisumu            22          23           45
Siaya             40          44           84
Homa Bay          27          10           37
Migori            19          23           42
Kakamega          27          29           56
Bungoma           25          53           78
Busia             39          40           79
Vihiga            14          12           26

STEP 2: Bayesian estimation (non-reversible MH / Gibbs, 3 chains)
  parameter      mean       sd  gelman_rubin_Rhat          ess     mcse  autocorr_lag1
    net_use -2.126720 0.089478           1.003104  1041.041952 0.002773       0.881947
  spray_use -1.824937 0.146577           1.000284  2449.522861 0.002962       0.950083
        Age  0.235018 0.036274           1.001569 12